# 01 — File Formats & Storage

**Skill focus:** Data Engineering (intern/co-op/new grad)

This notebook compares storage formats used in data pipelines:
- **JSON vs CSV vs Parquet vs Delta** — read/write performance, size
- **Columnar vs row storage** — scan patterns and when each wins
- **Compression** — Snappy, GZIP, Zstd tradeoffs
- **When to use each** — landing (JSON), staging (Parquet), serving (Delta)
- **Schema evolution** — how each format handles schema changes

In [ ]:
import time

from src.env import project_root, cache_path, BENCH_PATH, get_spark

spark = get_spark()
CACHE_PATH = cache_path("pokemon")

print(f"Root: {project_root}")
print(f"Cache: {CACHE_PATH}")
print(f"Bench: {BENCH_PATH}")

## 1. Load source data

Read Pokemon JSON from the ingestion cache (from notebook 00).

In [ ]:
df = spark.read.json(CACHE_PATH)
row_count = df.count()
print(f"Loaded {row_count} Pokemon records")
df.printSchema()

## 2. Write performance comparison

Write the same data to multiple formats and measure time and size.

In [ ]:
from pathlib import Path
Path(BENCH_PATH).mkdir(parents=True, exist_ok=True) if Path(BENCH_PATH).exists() or str(BENCH_PATH).startswith("/dbfs") else None

def write_and_measure(path: str, format_type: str, **kwargs) -> dict:
    """Write df to path, return elapsed_ms and size_mb."""
    full_path = f"{BENCH_PATH}/{format_type}"
    start = time.perf_counter()
    df.write.mode("overwrite").format(format_type).save(full_path)
    elapsed_ms = (time.perf_counter() - start) * 1000
    
    from pyspark.sql import SparkSession
    try:
        files = spark.sparkContext._jvm.org.apache.hadoop.fs.FileSystem.get(
            spark.sparkContext._jsc.hadoopConfiguration()
        ).listStatus(spark.sparkContext._jvm.org.apache.hadoop.fs.Path(full_path))
        size_bytes = sum(f.getLen() for f in files if f.isFile())
    except Exception:
        size_bytes = 0
    return {"format": format_type, "write_ms": round(elapsed_ms, 1), "size_mb": round(size_bytes / 1_000_000, 2)}

results = []
for fmt in ["json", "csv", "parquet", "delta"]:
    r = write_and_measure(BENCH_PATH, fmt)
    results.append(r)
    print(f"{fmt:10} write: {r['write_ms']:>8.1f} ms  size: {r['size_mb']:>6.2f} MB")

## 3. Read performance comparison

Read back from each format and measure time.

In [ ]:
def read_and_measure(fmt: str) -> float:
    path = f"{BENCH_PATH}/{fmt}"
    start = time.perf_counter()
    _ = spark.read.format(fmt).load(path).count()
    return (time.perf_counter() - start) * 1000

print("Read times (ms):")
for fmt in ["json", "csv", "parquet", "delta"]:
    r = read_and_measure(fmt)
    print(f"  {fmt:10} {r:>8.1f} ms")

## 4. Columnar vs row storage

**Row storage** (JSON, CSV): store rows one after another. Good for full-row scans.

**Columnar storage** (Parquet, Delta): store columns together. Good for analytical queries that select few columns.

**Demo:** Scan a single column vs full table — Parquet/Delta read much less I/O for columnar scans.

In [ ]:
# Parquet: selecting only 'id' reads fewer column chunks than selecting all columns
parquet_path = f"{BENCH_PATH}/parquet"

start_full = time.perf_counter()
spark.read.parquet(parquet_path).select("id", "name", "height", "weight").count()
full_ms = (time.perf_counter() - start_full) * 1000

start_single = time.perf_counter()
spark.read.parquet(parquet_path).select("id").count()
single_ms = (time.perf_counter() - start_single) * 1000

print("Parquet (columnar):")
print(f"  Full scan (4 cols): {full_ms:.1f} ms")
print(f"  Single column (id): {single_ms:.1f} ms")
print(f"  → Columnar: fewer columns = less I/O (predicate/column pushdown)")

## 5. Compression tradeoffs

Parquet supports multiple codecs. Snappy = fast, GZIP = smaller, Zstd = balanced.

In [ ]:
compressions = ["snappy", "gzip", "zstd"]
for codec in compressions:
    path = f"{BENCH_PATH}/parquet_{codec}"
    start = time.perf_counter()
    df.write.mode("overwrite").parquet(path, compression=codec)
    write_ms = (time.perf_counter() - start) * 1000
    # Size via spark
    size_mb = spark.read.parquet(path).inputFiles()
    # Approximate: use df's physical plan or file size
    import os
    try:
        total = sum(os.path.getsize(os.path.join(path, f)) for f in os.listdir(path) if f.endswith(".parquet"))
    except Exception:
        total = 0
    print(f"{codec:8} write: {write_ms:>6.1f} ms  (size varies by disk)")

In [ ]:
# Simpler: just show write times; size is format-dependent
print("Compression write times (ms):")
for codec in ["snappy", "gzip", "zstd"]:
    path = f"{BENCH_PATH}/parquet_{codec}"
    start = time.perf_counter()
    df.write.mode("overwrite").parquet(path, compression=codec)
    elapsed = (time.perf_counter() - start) * 1000
    print(f"  {codec:8} {elapsed:>6.1f} ms")
print("\nTypical: Snappy fastest, GZIP smallest, Zstd balanced.")

## 6. When to use each format

| Format | Use case | Why |
|--------|----------|-----|
| **JSON** | Landing / raw API | Human-readable, schema-flexible, easy to ingest |
| **CSV** | Ad-hoc exports | Universal, Excel-compatible |
| **Parquet** | Staging / analytics | Columnar, compressed, fast scans |
| **Delta** | Serving / production | ACID, time travel, schema enforcement, CDF |

## 7. Schema evolution

**JSON:** Schema-on-read. Add columns anytime; readers infer. No enforcement.

**Parquet:** Schema embedded. Add columns with `mergeSchema=True`. Dropping/renaming requires rewrite.

**Delta:** Schema evolution with `mergeSchema=True` or `ALTER TABLE`. Supports add column, change type (with constraints).

In [ ]:
# Delta schema evolution: add a column
delta_path = f"{BENCH_PATH}/delta"
from pyspark.sql.functions import lit

# Add a new column and write with mergeSchema
df_evolved = df.withColumn("benchmark_source", lit("01_file_formats"))
df_evolved.write.format("delta").mode("overwrite").option("mergeSchema", "true").save(delta_path)

evolved = spark.read.format("delta").load(delta_path)
print("Schema after adding 'benchmark_source':")
evolved.printSchema()

## Key takeaways

- **JSON** for landing (raw API responses)
- **Parquet** for staging and intermediate analytics (columnar, compressed)
- **Delta** for serving / production tables (ACID, time travel, schema evolution)
- **Compression:** Snappy for speed, GZIP for size, Zstd for balance